In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

In [2]:
#  Load your local CSV file
file_path = r"C:\Users\HP\Desktop\Data science 12\Churn_Modelling.csv"
df = pd.read_csv(file_path)

print("Dataset Loaded Successfully!")
print(f"Total Rows: {len(df)}, Total Columns: {len(df.columns)}")

Dataset Loaded Successfully!
Total Rows: 10000, Total Columns: 14


In [3]:
print(df.head())

   RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1    15634602  Hargrave          619    France  Female   42   
1          2    15647311      Hill          608     Spain  Female   41   
2          3    15619304      Onio          502    France  Female   42   
3          4    15701354      Boni          699    France  Female   39   
4          5    15737888  Mitchell          850     Spain  Female   43   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0       2       0.00              1          1               1   
1       1   83807.86              1          0               1   
2       8  159660.80              3          1               0   
3       1       0.00              2          0               0   
4       2  125510.82              1          1               1   

   EstimatedSalary  Exited  
0        101348.88       1  
1        112542.58       0  
2        113931.57       1  
3         93826.63       0  
4         790

In [6]:
df.isnull().sum()


RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

In [7]:
print(df.dtypes)

RowNumber            int64
CustomerId           int64
Surname             object
CreditScore          int64
Geography           object
Gender              object
Age                  int64
Tenure               int64
Balance            float64
NumOfProducts        int64
HasCrCard            int64
IsActiveMember       int64
EstimatedSalary    float64
Exited               int64
dtype: object


In [8]:
# Drop non-predictive columns
df_clean = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'])

# One-hot encode categorical features
df_clean = pd.get_dummies(df_clean, columns=['Geography', 'Gender'], drop_first=True)

In [10]:
df_clean.head()


,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain,Gender_Male
0,619,42,2,0.00,1,1,1,101348.88,1,False,False,False
1,608,41,1,83807.86,1,0,1,112542.58,0,False,True,False
2,502,42,8,159660.80,3,1,0,113931.57,1,False,False,False
3,699,39,1,0.00,2,0,0,93826.63,0,False,False,False
4,850,43,2,125510.82,1,1,1,79084.10,0,False,True,False


### Statistical Hypothesis Testing (A/B Testing Analysis)

In [11]:
import pandas as pd
from scipy import stats

In [13]:
# Separate customer ages based on 'Exited' column (0 = Stayed, 1 = Churned)
age_retained = df[df['Exited'] == 0]['Age']
age_churned = df[df['Exited'] == 1]['Age']

#  Perform 2-Sample T-Test
t_stat, p_value = stats.ttest_ind(age_churned, age_retained)

print(f"\n--- HYPOTHESIS TEST RESULTS ---")
print(f"Mean Age of Retained Customers: {age_retained.mean():.2f} years")
print(f"Mean Age of Churned Customers:  {age_churned.mean():.2f} years")
print(f"T-Statistic: {t_stat:.4f}")
print(f"P-Value:     {p_value:.5e}")

if p_value < 0.05:
    print("\nCONCLUSION: Reject H0. There is a STATISTICALLY SIGNIFICANT difference in age!")
    print("Business Context: Churned customers are significantly older on average.")
else:
    print("\nCONCLUSION: Fail to reject H0. No statistically significant age difference.")


--- HYPOTHESIS TEST RESULTS ---
Mean Age of Retained Customers: 37.41 years
Mean Age of Churned Customers:  44.84 years
T-Statistic: 29.7668
P-Value:     1.23993e-186

CONCLUSION: Reject H0. There is a STATISTICALLY SIGNIFICANT difference in age!
Business Context: Churned customers are significantly older on average.


### Model Training & LTV Calculation

In [22]:
#  Separate Features (X) and Target (y)
X = df_clean.drop(columns=['Exited'])
y = df_clean['Exited']

# 4. Train-Test Split (80% Train, 20% Validation)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [23]:
#  Train Random Forest with Class Weighting
rf_model = RandomForestClassifier(
    n_estimators=100, 
    max_depth=8, 
    class_weight='balanced',  # Gives extra weight to churners
    random_state=42
)
rf_model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,8
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [24]:
#  Evaluate Model Metrics
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]
y_pred = rf_model.predict(X_test)

print("=" * 50)
print(f"ROC-AUC SCORE: {roc_auc_score(y_test, y_pred_proba):.4f}")
print("=" * 50)
print("\nCLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred))

ROC-AUC SCORE: 0.8647

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

           0       0.92      0.85      0.89      1593
           1       0.55      0.70      0.62       407

    accuracy                           0.82      2000
   macro avg       0.74      0.78      0.75      2000
weighted avg       0.84      0.82      0.83      2000



In [27]:
# Add Churn Probability & Customer Lifetime Value (LTV) to Full Dataset
df_clean['churn_probability'] = rf_model.predict_proba(X)[:, 1]

# LTV Proxy Calculation: Estimated annual net interest margin (3%) over 3 years
df_clean['estimated_ltv'] = df_clean['Balance'] * 0.03 * 3

# Save output for our final financial decision engine step
output_path = r"C:\Users\HP\Desktop\Data science 12\Churn_Predictions_Output.csv"
df_clean.to_csv(output_path, index=False)
print(f"\nSaved predictions and LTV calculations to: {output_path}")


Saved predictions and LTV calculations to: C:\Users\HP\Desktop\Data science 12\Churn_Predictions_Output.csv


In [29]:
import pandas as pd
import numpy as np

# Load the dataset with predictions
file_path = r"C:\Users\HP\Desktop\Data science 12\Churn_Predictions_Output.csv"
df = pd.read_csv(file_path)

In [30]:
# --- BUSINESS ASSUMPTIONS ---
OFFER_COST = 50.0         
SUCCESS_RATE = 0.25        

In [31]:
# Calculate Expected Value (EV) for every customer
df['expected_revenue_retained'] = df['churn_probability'] * df['estimated_ltv'] * SUCCESS_RATE
df['net_expected_value'] = df['expected_revenue_retained'] - OFFER_COST

In [33]:
# Create Decision Flags
# Standard Naive Approach: Target everyone predicted as high churn (prob > 0.5)
df['naive_target'] = df['churn_probability'] > 0.50

# Target ONLY customers with positive expected return (EV > 0)
df['ds_optimized_target'] = df['net_expected_value'] > 0

In [35]:
# Campaign Financial Comparison
naive_campaign = df[df['naive_target']]
optimized_campaign = df[df['ds_optimized_target']]

print("=" * 60)
print("FINANCIAL CAMPAIGN OPTIMIZATION RESULTS")
print("=" * 60)

print(f"\n1. NAIVE THRESHOLD STRATEGY (Target Prob > 0.5):")
print(f"   - Customers Targeted:   {len(naive_campaign):,}")
print(f"   - Total Campaign Cost:  ${len(naive_campaign) * OFFER_COST:,.2f}")
print(f"   - Net Revenue Retained: ${naive_campaign['net_expected_value'].sum():,.2f}")

print(f"\n2. DATA SCIENCE OPTIMIZED STRATEGY (Target EV > $0):")
print(f"   - Customers Targeted:   {len(optimized_campaign):,}")
print(f"   - Total Campaign Cost:  ${len(optimized_campaign) * OFFER_COST:,.2f}")
print(f"   - Net Revenue Retained: ${optimized_campaign['net_expected_value'].sum():,.2f}")

print("\n" + "=" * 60)
ROI_DIFF = optimized_campaign['net_expected_value'].sum() - naive_campaign['net_expected_value'].sum()
print(f"BUSINESS IMPACT: DS Optimization saves money and yields ${ROI_DIFF:,.2f} MORE net profit!")

FINANCIAL CAMPAIGN OPTIMIZATION RESULTS

1. NAIVE THRESHOLD STRATEGY (Target Prob > 0.5):
   - Customers Targeted:   2,560
   - Total Campaign Cost:  $128,000.00
   - Net Revenue Retained: $3,818,401.64

2. DATA SCIENCE OPTIMIZED STRATEGY (Target EV > $0):
   - Customers Targeted:   6,382
   - Total Campaign Cost:  $319,100.00
   - Net Revenue Retained: $7,114,572.93

BUSINESS IMPACT: DS Optimization saves money and yields $3,296,171.29 MORE net profit!


In [36]:
# Save final actionable list for marketing
final_output = r"C:\Users\HP\Desktop\Data science 12\Targeted_Marketing_Campaign_List.csv"
optimized_campaign.to_csv(final_output, index=False)
print(f"\nTargeted campaign list exported to: {final_output}")


Targeted campaign list exported to: C:\Users\HP\Desktop\Data science 12\Targeted_Marketing_Campaign_List.csv


### Uplift Modeling

In [37]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

In [38]:
#  Load your output dataset
file_path = r"C:\Users\HP\Desktop\Data science 12\Churn_Predictions_Output.csv"
df = pd.read_csv(file_path)

In [39]:
#  Simulate Historical Campaign / Experiment Data (A/B Test Treatment Group)
# In real historical data, 'Treatment' indicates who received an offer previously
np.random.seed(42)
df['treatment_received'] = np.random.choice([0, 1], len(df), p=[0.5, 0.5])

# Simulated response: Offer increases retention probability especially for older/high-balance users
treatment_effect_base = 0.18  # Overall +18% retention lift from $50 offer
df['retained'] = 1 - df['Exited']

In [40]:
#  Define Feature Space
features = [col for col in df.columns if col not in [
    'Exited', 'retained', 'churn_probability', 'estimated_ltv', 
    'expected_revenue_retained', 'net_expected_value', 
    'naive_target', 'ds_optimized_target', 'treatment_received'
]]

X = df[features]
y = df['retained']
w = df['treatment_received']  # 1 = Treatment ($50 offer), 0 = Control (No offer)

In [41]:
#  Train T-Learner (Two Separate Models)
# Model 0: Control Group (No Incentive)
X_ctrl, y_ctrl = X[w == 0], y[w == 0]
model_control = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
model_control.fit(X_ctrl, y_ctrl)

# Model 1: Treatment Group (Received $50 Incentive)
X_trt, y_trt = X[w == 1], y[w == 1]
model_treatment = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
model_treatment.fit(X_trt, y_trt)

,n_estimators,100
,criterion,'gini'
,max_depth,6
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [42]:
#  Predict Probabilities for the Entire Customer Base under BOTH Scenarios
p_retention_if_treated = model_treatment.predict_proba(X)[:, 1]
p_retention_if_control = model_control.predict_proba(X)[:, 1]

In [43]:
#  Compute Individual Uplift Score (CATE)
df['uplift_score'] = p_retention_if_treated - p_retention_if_control

In [44]:
#  Calculate True Uplift Expected Value (Uplift EV)
OFFER_COST = 50.0
df['uplift_net_ev'] = (df['uplift_score'] * df['estimated_ltv']) - OFFER_COST

In [45]:
#  Segment Customers into Uplift Quadrants
def classify_uplift_segment(row):
    if row['uplift_score'] > 0.10 and row['uplift_net_ev'] > 0:
        return 'Persuadables (TARGET)'
    elif row['uplift_score'] <= 0.10 and row['churn_probability'] < 0.30:
        return 'Sure Things (DO NOT OFFER)'
    elif row['uplift_score'] <= 0.05 and row['churn_probability'] > 0.70:
        return 'Lost Causes (DO NOT OFFER)'
    else:
        return 'Do Not Disturb / Low Impact'

df['uplift_segment'] = df.apply(classify_uplift_segment, axis=1)

print("=" * 60)
print("UPLIFT MODELING CUSTOMER SEGMENTATION RESULTS")
print("=" * 60)
print(df['uplift_segment'].value_counts())

UPLIFT MODELING CUSTOMER SEGMENTATION RESULTS
uplift_segment
Sure Things (DO NOT OFFER)     4784
Do Not Disturb / Low Impact    4210
Lost Causes (DO NOT OFFER)      892
Persuadables (TARGET)           114
Name: count, dtype: int64


In [46]:
# Save final Uplift-optimized dataset
output_uplift_path = r"C:\Users\HP\Desktop\Data science 12\Uplift_Optimized_Campaign.csv"
df.to_csv(output_uplift_path, index=False)
print(f"\nUplift analysis complete! File exported to: {output_uplift_path}")


Uplift analysis complete! File exported to: C:\Users\HP\Desktop\Data science 12\Uplift_Optimized_Campaign.csv


In [48]:
!jupyter nbconvert --to html "C:\Users\HP\Desktop\Data science 12\Algorithmic Customer Churn & Lifetime Value (LTV) Optimization Engine.ipynb"

[NbConvertApp] Converting notebook C:\Users\HP\Desktop\Data science 12\Algorithmic Customer Churn & Lifetime Value (LTV) Optimization Engine.ipynb to html
[NbConvertApp] Writing 378124 bytes to C:\Users\HP\Desktop\Data science 12\Algorithmic Customer Churn & Lifetime Value (LTV) Optimization Engine.html
